[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/contact_geometry/contact_geometry.ipynb)

# Legendre Transformation and Contact Geometry

This example computes the Legendre transformation of an ideal gas from its internal energy as a function of entropy to its Helmholtz free energy as a function of temperature. Controlling temperature in the laboratory is much easier than holding entropy fixed, and the Helmholtz free energy (internal energy minus temperature times entropy) is the potential suited to controlled temperature. Geometrically, each tangent line to the internal energy curve has the temperature as its slope and meets zero entropy at the Helmholtz free energy, so the family of tangent intercepts traces the Helmholtz free energy curve.

The five thermodynamic variables of the gas (internal energy, entropy, volume, temperature, and pressure) are modelled as components of an antivector in the exterior space `u0 s0 v0 t0 p0`, read by pairing (`t & point`). Equilibrium embeds each state of entropy and volume as a point on a two-dimensional surface in that five-dimensional space. Along the surface, the contact covector `contact_element(point)` pairs to zero with every tangent step (`contact_element(point) & tangent == 0`) because a change of internal energy equals heat added minus mechanical work done. The Legendre transformation maps the ambient space to itself, and pulling the contact covector at the transformed point back through `.adjugate()` of the map's pushforward recovers the original contact covector, carrying equilibrium surfaces to equilibrium surfaces. Wedging the contact derivative `pv - ts` twice with the contact covector gives a nonvanishing five-blade everywhere, showing that the contact hyperplanes twist throughout the space and admit only two-dimensional equilibrium surfaces.

In differential-form and Jacobian-matrix notation on $(U, S, V, T, p)$, the contact one-form $\theta = dU - T\,dS + p\,dV$ defines Legendrian equilibrium surfaces by $\theta = 0$ with non-integrability $\theta \wedge d\theta \wedge d\theta \neq 0$, and the Helmholtz map $\Phi(U, S, V, T, p) = (U - TS,\, T,\, V,\, -S,\, p)$ preserves $\theta$ through the transposed-Jacobian pullback $(D\Phi)^{\mathsf T}(1, -\tilde T, \tilde p, 0, 0)^{\mathsf T} = (1, -T, p, 0, 0)^{\mathsf T}$, rewriting $dU = T\,dS - p\,dV$ as $dF = -S\,dT - p\,dV$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext
from examples.util.animation import show_states
from examples.thermodynamics.contact_geometry import render

# Five-dimensional exterior contact space of energy, entropy, volume, temperature, and pressure.
ga = Algebra("u0s0v0t0p0")
mv = NumpyContext(ga, dtype=np.float64).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Ambient = ga.gatype.antivector()                                              # Antivector-
# States in the two-dimensional subspace of entropy and volume.
State = ga.gatype(mv.s.dual() + mv.v.dual())                                  # Antivector-
Potential = ga.gatype.vector()                                                # Vector+
Pushforward = ga.gatype((Ambient, Ambient))

# Exterior derivative of the contact covector, pairing temperature with entropy against pressure with volume.
contact_derivative = mv.pv - mv.ts                                            # [] Bivector+

## Equilibrium in five variables

An ideal gas's internal energy depends exponentially on entropy and as an inverse power on volume; differentiating gives temperature along entropy and minus pressure along volume, lifting each `(S, V)` state to a point in the five-dimensional space. Differentiating that embedding along entropy at fixed volume gives the section tangents, each pairing to zero with `contact_element(points)`.

In [ ]:
# Dimensionless heat capacity at fixed volume, sample counts, and constant-volume section parameters.
heat_capacity = 1.5
n_entropies, n_volumes = 93, 5
section_volume, frame_duration_ms = 1.0, 40
entropies = np.linspace(-0.8, 1.2, n_entropies)
volumes = np.array([0.6, 0.8, 1.0, 1.3, 1.7])                                 # [n_volumes]

In [ ]:
def internal_energy(state: State) -> Scalar:
    """Internal energy of an ideal gas with constant heat capacity at fixed volume."""
    return heat_capacity * (((mv.s & state) - (mv.v & state).log()) / heat_capacity).exp()


def equilibrium(state: State) -> Ambient:
    """Equilibrium point in the five-dimensional space for each state of entropy and volume."""
    energies = internal_energy(state)                                         # [...] Scalar+
    temperatures = energies / heat_capacity                                   # [...] Scalar+
    pressures = temperatures / (mv.v & state)                                 # [...] Scalar+
    return state + mv.u.dual() * energies + mv.t.dual() * temperatures + mv.p.dual() * pressures


def entropy_tangent(state: State) -> Ambient:
    """Tangent vector to the equilibrium surface along a unit step of entropy at fixed volume."""
    energies = internal_energy(state)                                         # [...] Scalar+
    temperatures = energies / heat_capacity                                   # [...] Scalar+
    pressures = temperatures / (mv.v & state)                                 # [...] Scalar+
    return (mv.s.dual() + mv.u.dual() * temperatures
            + (mv.t.dual() * temperatures + mv.p.dual() * pressures) / heat_capacity)


def contact_element(point: Ambient) -> Potential:
    """Contact covector whose pairing with a step vanishes when energy change equals heat minus work."""
    return mv.u - (mv.t & point) * mv.s + (mv.p & point) * mv.v               # [...] Potential+


family_states = mv.s.dual() * entropies[None, :] + mv.v.dual() * volumes[:, None]   # [n_volumes, n_entropies] State-
family = equilibrium(family_states)                                                 # [n_volumes, n_entropies] Ambient-
section_states = mv.s.dual() * entropies + mv.v.dual() * section_volume             # [n_entropies] State-
points = equilibrium(section_states)                                                # [n_entropies] Ambient-
tangents = entropy_tangent(section_states)                                          # [n_entropies] Ambient-
render.draw_energy(family, points, tangents);

Green curves show isochores of internal energy against entropy, with the unit-volume section drawn in bold ink and enveloped by faint grey-blue tangent lines. The orange tangent at the midpoint has the local temperature as its slope and meets the vertical axis at zero entropy at the Helmholtz free energy `F`.

## The Legendre transformation

The Helmholtz Legendre map replaces internal energy with `energy - temperature * entropy` while swapping entropy and temperature with a minus sign that preserves the contact form. Its linear pushforward carries tangent vectors onto the transformed curve, and pulling `contact_element(transformed_points)` back through `pushforwards.adjugate()` reproduces `contact_element(points)` identically.

In [ ]:
unchanged_mechanical = mv.v.dual() * (mv.v & Ambient) + mv.p.dual() * (mv.p & Ambient)   # [] Ambient- <- Ambient-
entropy_temperature_swap = mv.s.dual() * (mv.t & Ambient) - mv.t.dual() * (mv.s & Ambient)   # [] Ambient- <- Ambient-


def helmholtz(point: Ambient) -> Ambient:
    """Legendre image replacing internal energy with Helmholtz free energy and swapping entropy with temperature."""
    entropy, temperature = mv.s & point, mv.t & point                         # [...] Scalar+, Scalar+
    return (mv.u.dual() * ((mv.u & point) - temperature * entropy) + mv.s.dual() * temperature - mv.t.dual() * entropy
            + mv.v.dual() * (mv.v & point) + mv.p.dual() * (mv.p & point))


def helmholtz_pushforward(point: Ambient) -> Pushforward:
    """Linear pushforward of ambient tangent vectors under the Helmholtz Legendre transformation."""
    entropy, temperature = mv.s & point, mv.t & point                         # [...] Scalar+, Scalar+
    energy_step = (mv.u - entropy * mv.t - temperature * mv.s) & Ambient      # [...] Scalar+ <- Ambient-
    return mv.u.dual() * energy_step + entropy_temperature_swap + unchanged_mechanical   # [...] Ambient- <- Ambient-


transformed_family = helmholtz(family)                                        # [n_volumes, n_entropies] Ambient-
transformed_points = helmholtz(points)                                        # [n_entropies] Ambient-
pushforwards = helmholtz_pushforward(points)                                  # [n_entropies] Ambient- <- Ambient-
transformed_tangents = pushforwards(tangents)                                 # [n_entropies] Ambient-
render.draw(family, transformed_family, points, tangents, transformed_points, transformed_tangents);

In the right chart, the transformed section plots Helmholtz free energy against temperature, where the orange tangent has minus the entropy as its slope and meets zero temperature at the internal energy `U`.

## Sweeping the tangent

As the point of tangency moves along the constant-volume curve, its tangent steepens with rising temperature on the left while its zero-entropy intercept traces out the concave Helmholtz free energy curve on the right.

In [ ]:
def tangent_stages():
    """Yields each point of tangency and its tangent vector along the section before and after the Legendre map."""
    for index in range(n_entropies):
        yield points[index], tangents[index], transformed_points[index], transformed_tangents[index]


stages = show_states(tangent_stages(), render.legendre, family, transformed_family, points, tangents,
                     transformed_points, transformed_tangents, name="thermodynamics_legendre",
                     duration_ms=frame_duration_ms, n_frames=n_entropies)

In [ ]:
# checks
np.testing.assert_allclose((contact_element(points) & tangents).kernel, 0, atol=1e-10)
np.testing.assert_allclose((contact_element(transformed_points) & transformed_tangents).kernel, 0, atol=1e-10)
# Pulling the contact covector back through the adjugate of the pushforward recovers the original contact covector.
pulled_back = pushforwards.adjugate()(contact_element(transformed_points))    # [n_entropies] Potential+
np.testing.assert_allclose((pulled_back - contact_element(points)).kernel, 0, atol=1e-10)
# On the transformed curve, the slope of Helmholtz free energy with respect to temperature equals minus the entropy.
np.testing.assert_allclose(((mv.u & transformed_tangents) / (mv.s & transformed_tangents) + (mv.s & points)).kernel,
                           0, atol=1e-10)
# Wedging the contact derivative twice with the contact covector produces a nonvanishing five-blade everywhere.
volume_forms = contact_element(points) ^ contact_derivative ^ contact_derivative   # [n_entropies] Pseudoscalar+
assert np.all(volume_forms.dual().abs() > 1)